In [ ]:
import time
import random

def get_embedding_with_retry(text, model, max_retries=5, extra_params=None):
    """
    دریافت embedding با مدیریت Rate Limiting (Exponential Backoff)
    Fetch an embedding with rate-limit retry (exponential backoff + jitter)
    """
    extra_params = extra_params or {}
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(
                model=model,
                input=text,
                **extra_params,
            )
            return response.data[0].embedding
        except Exception as e:
            error_message = str(e)
            if "429" in error_message or "rate" in error_message.lower():
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"   ⏳ Rate limited, waiting {wait_time:.1f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"   ❌ Unexpected error: {error_message}")
                return None
    print(f"   ❌ Failed after {max_retries} attempts")
    return None


def cosine_similarity(a, b):
    """محاسبهٔ شباهت کسینوسی / Cosine similarity between two vectors."""
    import numpy as np
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


The Qdrant section in this notebook covers five important topics:
1. Connecting to Qdrant and creating a collection.
2. Storing embeddings using upsert.
3. Performing semantic similarity searches.
4. Filtering results using payloads and indexing.
5. Hybrid search (Dense + Sparse) and RRF.
6. Running Qdrant using Docker instead of in-memory storage.

In [ ]:

!pip install -q openai pinecone qdrant-client google-genai numpy


In [ ]:
import os

AVALAI_API_KEY = os.environ.get("AVALAI_API_KEY", "YOUR_AVALAI_API_KEY")
AVALAI_BASE_URL = "https://api.avalai.ir/v1"

PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "YOUR_PINECONE_API_KEY")
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL_OPENAI = "text-embedding-3-small"

In [ ]:
from qdrant_client import QdrantClient

``QdrantClient`` is the main Python interface for communicating with Qdrant.

Through this object, we can:
- Create collections.
- Insert vectors.
- Search vectors.
- Update and delete records.
- Apply filters.

In [ ]:
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range,
)
import uuid   # Universally unique identifier

These are data structures that define how we interact with Qdrant.
- Distance:	Defines how vector similarity is calculated.
- VectorParams:	Defines vector configuration.
- PointStruct:	Represents a record stored in Qdrant.
- Filter:	Defines search conditions.
- FieldCondition:	Specifies which payload field to filter.
- MatchValue:	Matches an exact value.
- Range:	Defines numerical ranges.

In [ ]:
qdrant = QdrantClient(":memory:")

- In-memory: Data exists in RAM and is temporary.
- Server/Docker: Qdrant runs as a separate service and can persist data on disk.

In [ ]:
COLLECTION_NAME = "university_FAQ"
qdrant.create_collection(collection_name=COLLECTION_NAME, vectors_config=VectorParams(size=1536, distance=Distance.COSINE))

what is a collection?

Think of a collection as a table in a traditional relational database.

For example:
- MySQL → Table
- PostgreSQL → Table
- Qdrant → Collection
However, a Qdrant collection is designed to store vector-based records.

**Very important**: The collection's vector dimension must match the dimension of the embedding model output.
___________________________________________________________________

## Inserting Data into Qdrant (Upsert)

In [ ]:
COURSE_FAQS = [
    {"id": "faq_1", "question": "شرایط وام دانشجویی چیست؟",
     "answer": "دانشجویان مشمول وام باید حداقل ۱۲ واحد گذرانده باشند و معدل بالای ۱۲ داشته باشند.",
     "category": "financial_aid", "level": "undergraduate"},
    {"id": "faq_2", "question": "چگونه می‌توانم یک درس را حذف کنم؟",
     "answer": "حذف واحد تا هفته سوم ترم امکان‌پذیر است، از طریق سامانه آموزشی.",
     "category": "registration", "level": "undergraduate"},
    {"id": "faq_3", "question": "شرایط پذیرش دکتری چیست؟",
     "answer": "پذیرش دکتری نیازمند مصاحبه علمی و حداقل معدل ۱۶ در مقطع کارشناسی ارشد است.",
     "category": "admissions", "level": "phd"},
    {"id": "faq_4", "question": "چگونه می‌توانم برای فرصت مطالعاتی درخواست بدهم؟",
     "answer": "دانشجویان دکتری با تایید استاد راهنما می‌توانند برای فرصت مطالعاتی خارج از کشور اقدام کنند.",
     "category": "academic", "level": "phd"},
    {"id": "faq_5", "question": "مهلت ثبت‌نام در کلاس‌ها چه زمانی است؟",
     "answer": "ثبت‌نام معمولاً دو هفته قبل از شروع ترم آغاز می‌شود.",
     "category": "registration", "level": "undergraduate"},
]
points = []
for i, faq in enumerate(COURSE_FAQS, start=1):
  embeding = get_embedding_with_retry(faq["question"], model=EMBEDDING_MODEL_OPENAI, max_retries=5, extra_params=None):
  if embeding:

    points.append(PointStruct(id=i, vector=embeding, payload={
        "question": faq["question"],
        "answer": faq["answer"],
        "category": faq["category"],
        "level": faq["level"],

    }))

    time.sleep(0.5)

qdrant.upsert(
    collection_name = COLLECTION_NAME,
    points = points,
    wait=True
)

Why do we need payload?

Because the vector itself doesn't contain the original readable question and answer.

We need payload to retrieve that information after finding a similar vector.

Also, Qdrant allows us to filter records using payload fields.

Note : ``upsert`` means:
- If the point ID doesn't exist, insert it.
- If the point ID already exists, replace/update that point.

In [ ]:
def qudrant_search(query, top_k=3, query_filter=None):
  query_vec = get_embedding_with_retry(query, model=EMBEDDING_MODEL_OPENAI) # Convert the user's question into an embedding

  # Search inside the collection
  hits = qudrant.query_points(
      query=query_vec, collection_name=QDRANT_COLLECTION, limit=top_k,
      with_payload=True,    # we want the original FAQ question and answer, not just its vector.
      query_filter=query_filter,).points  # Optional filtering condition
  for hit in hits:
    print(f"  score={hit.score:.4f}  Q: {hit.payload['question']}")
    print(f"             A: {hit.payload['answer']}")

return hits


## Filtering with Payload

In [ ]:
# How to get a loan- undergrads
_ = qdrant_search(
    "چطور وام بگیرم",
    query_filter=Filter(must=[FieldCondition(key="level", match=MatchValue(value="undergraduate"))]),
)

``must`` : all specified conditions must be satisfied.

In [ ]:
# Payload indexing is for efficient metadata filtering.
qdrant.create_payload_index(
    collection_name=QDRANT_COLLECTION,
    field_name="category",
    field_schema="keyword",
)

## Hybrid Search: Dense + Sparse

- Dense retrieval uses embeddings.
  - Understands semantic similarity
  - Useful for paraphrasing
  - Embedding-based
- Sparse retrieval focuses more on matching terms and their importance.
  - Helps with exact terms
  - Useful for specific keywords
  - Term-based representation

In [ ]:
from collections import Counter  # Counter counts how many times each word appears.

from qdrant_client.models import (
    SparseVector,
    SparseVectorParams
)

SPARSE_VOCAB_SIZE = 2000  # the size of the artificial vocabulary space used by the hashing function

In [ ]:
def simple_sparse_vector(
    text,
    vocab_size=SPARSE_VOCAB_SIZE
):

    words = text.lower().split()

    counts = Counter(
        hash(w) % vocab_size
        for w in words
    )

    indices = list(counts.keys())

    values = [
        float(v)
        for v in counts.values()
    ]

    return SparseVector(
        indices=indices,
        values=values
    )

``` bash
text = "delete course course"
["delete", "course", "course"]
delete → 125
course → 845

{
    125: 1,
    845: 2
}

indices = [125, 845]
values = [1.0, 2.0]
```
Note : Unlike a dense vector, which contains a value for every dimension, a sparse vector stores only non-zero entries.

## Creating a Hybrid Collection

In [ ]:
HYBRID_COLLECTION = "course_hybrid_demo"

qdrant.create_collection(
    collection_name=HYBRID_COLLECTION,

    vectors_config={
        "dense": VectorParams(
            size=1536,
            distance=Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    },
)

In [ ]:
# create both vectors for each FAQ
dense_vec = get_embedding_with_retry(
    faq["question"],
    model=EMBEDDING_MODEL_OPENAI
)

sparse_vec = simple_sparse_vector(
    faq["question"]
)

In [ ]:
# store them together
PointStruct(
    id=i,

    vector={
        "dense": dense_vec,
        "sparse": sparse_vec
    },

    payload={
        "question": faq["question"],
        "answer": faq["answer"]
    },
)

In [ ]:
query_text = "حذف واحد درسی"

dense_query = get_embedding_with_retry(
    query_text,
    model=EMBEDDING_MODEL_OPENAI
)

sparse_query = simple_sparse_vector(query_text)

we generate two query representations:
- dense_query: Semantic embedding.
- sparse_query: Sparse keyword representation.

In [ ]:
# RRF — Reciprocal Rank Fusion
hybrid_result = qdrant.query_points(
    collection_name=HYBRID_COLLECTION,

    prefetch=[
        models.Prefetch(query=sparse_query, using="sparse", limit=10),  # Retrieves the top 10 results using sparse retrieval.

        models.Prefetch(query=dense_query, using="dense", limit=10), # Retrieves the top 10 results using dense retrieval.
    ],

    query=models.RrfQuery(rrf=models.Rrf()),  # RRF combines the rankings produced by the two retrieval methods.

    limit=3,
).points